In [28]:
#importing the Libraies
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


In [29]:
dataset=pd.read_csv("Social_Network_Ads.csv")

In [30]:
dataset

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0
...,...,...,...,...,...
395,15691863,Female,46,41000,1
396,15706071,Male,51,23000,1
397,15654296,Female,50,20000,1
398,15755018,Male,36,33000,0


In [31]:
dataset=pd.get_dummies(dataset,dtype=int,drop_first='True')

In [32]:
dataset

,User ID,Age,EstimatedSalary,Purchased,Gender_Male
0,15624510,19,19000,0,1
1,15810944,35,20000,0,1
2,15668575,26,43000,0,0
3,15603246,27,57000,0,0
4,15804002,19,76000,0,1
...,...,...,...,...,...
395,15691863,46,41000,1,0
396,15706071,51,23000,1,1
397,15654296,50,20000,1,0
398,15755018,36,33000,0,1


In [33]:
dataset=dataset.drop("User ID",axis=1)

In [34]:
dataset["Purchased"].value_counts()

Purchased
0    257
1    143
Name: count, dtype: int64

In [35]:
indep=dataset[["Age","EstimatedSalary","Gender_Male"]]
dep=dataset["Purchased"]


In [36]:
indep.shape

(400, 3)

In [37]:
dep

0      0
1      0
2      0
3      0
4      0
      ..
395    1
396    1
397    1
398    0
399    1
Name: Purchased, Length: 400, dtype: int64

In [38]:
#split into training set and test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(indep, dep, test_size = 1/3, random_state = 0)

In [39]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [40]:
from sklearn.tree import DecisionTreeClassifier

In [41]:
#https://scikit-learn.org/stable/modules/model_evaluation.html#scoring-parameter

In [42]:
from sklearn.model_selection import GridSearchCV

param_grid = {'criterion':['gini','entropy'],
              'max_features': ['sqrt', 'log2', None],
              'splitter':['best','random']} 



grid = GridSearchCV(DecisionTreeClassifier(),param_grid,refit=True,verbose=3,n_jobs=-1,scoring='f1_weighted',cv=5)
   
# fitting the model for grid search 
grid.fit(X_train, y_train) 
 



Fitting 5 folds for each of 12 candidates, totalling 60 fits


,estimator,DecisionTreeClassifier()
,param_grid,"{'criterion': ['gini', 'entropy'], 'max_features': ['sqrt', 'log2', ...], 'splitter': ['best', 'random']}"
,scoring,'f1_weighted'
,n_jobs,-1
,refit,True
,cv,5
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'entropy'


In [43]:
# print best parameter after tuning 
#print(grid.best_params_) 
re=grid.cv_results_
#print(re)
grid_predictions = grid.predict(X_test) 
   

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, grid_predictions)



# print classification report 
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, grid_predictions)



In [44]:
from sklearn.metrics import f1_score
f1_macro=f1_score(y_test,grid_predictions,average='weighted')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)


The f1_macro value for best parameter {'criterion': 'entropy', 'max_features': None, 'splitter': 'best'}: 0.8805970149253731


In [45]:
print("The confusion Matrix:\n",cm)

The confusion Matrix:
 [[77  8]
 [ 8 41]]


In [46]:
print("The report:\n",clf_report)

The report:
               precision    recall  f1-score   support

           0       0.91      0.91      0.91        85
           1       0.84      0.84      0.84        49

    accuracy                           0.88       134
   macro avg       0.87      0.87      0.87       134
weighted avg       0.88      0.88      0.88       134



In [47]:
from sklearn.metrics import roc_auc_score

roc_auc_score(y_test,grid.predict_proba(X_test)[:,1])


0.8713085234093638

In [48]:
table=pd.DataFrame.from_dict(re)

In [49]:
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_features,param_splitter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.005862,0.001987,0.009696,0.002511,gini,sqrt,best,"{'criterion': 'gini', 'max_features': 'sqrt', ...",0.782557,0.868752,0.796284,0.925272,0.923510,0.859275,0.060704,2
1,0.005233,0.000993,0.011332,0.001633,gini,sqrt,random,"{'criterion': 'gini', 'max_features': 'sqrt', ...",0.808927,0.849057,0.759721,0.851527,0.773585,0.808563,0.037669,12
2,0.005033,0.000535,0.011723,0.000800,gini,log2,best,"{'criterion': 'gini', 'max_features': 'log2', ...",0.847141,0.870047,0.756031,0.906166,0.844370,0.844751,0.049576,4
3,0.004771,0.000462,0.011582,0.000171,gini,log2,random,"{'criterion': 'gini', 'max_features': 'log2', ...",0.829615,0.832918,0.753180,0.869709,0.865054,0.830095,0.041751,9
4,0.004080,0.000257,0.010592,0.000736,gini,None,best,"{'criterion': 'gini', 'max_features': None, 's...",0.829615,0.849057,0.814409,0.870362,0.924528,0.857594,0.038377,3
5,0.003954,0.000663,0.010192,0.001218,gini,None,random,"{'criterion': 'gini', 'max_features': None, 's...",0.826263,0.852030,0.775815,0.868632,0.826499,0.829848,0.031432,10
6,0.003634,0.000576,0.008837,0.000470,entropy,sqrt,best,"{'criterion': 'entropy', 'max_features': 'sqrt...",0.804764,0.811321,0.793565,0.886792,0.885265,0.836341,0.040968,5
7,0.003451,0.000490,0.009825,0.001304,entropy,sqrt,random,"{'criterion': 'entropy', 'max_features': 'sqrt...",0.847141,0.849057,0.703055,0.887907,0.885265,0.834485,0.067940,7
8,0.002954,0.000428,0.009335,0.001858,entropy,log2,best,"{'criterion': 'entropy', 'max_features': 'log2...",0.782557,0.832918,0.793565,0.870362,0.883278,0.832536,0.040056,8
9,0.003857,0.000839,0.009315,0.001109,entropy,log2,random,"{'criterion': 'entropy', 'max_features': 'log2...",0.759544,0.773585,0.851527,0.847020,0.862442,0.818824,0.043192,11
